In [1]:
import numpy as np

In [4]:
x = np.zeros((4,5))
x.shape[1]

5

In [8]:
class SelfAttention:
    def __init__(self, dim_in, dim_out):
        # dim_in = embedding dim 
        # dim_out = size of Q, K and V vectors

        self.Wq = np.random.randn(dim_in, dim_out)
        self.Wk = np.random.randn(dim_in, dim_out)
        self.Wv = np.random.randn(dim_in, dim_out)
    
    def forward(self, x):
        Q = x @ self.Wq
        K = x @ self.Wk
        V = x @ self.Wv
        T = x.shape[0]
        # we gotta mask the future to into -inf do that softmax returns 0 and thereby future is hidden for the model

        lower_triangle_mask = np.tril(np.ones((T,T))) # creates upper tri of 1 and lower tri of 0
        inner = (Q @ K.T)/np.sqrt(self.Wk.shape[1])
        inner = np.where(lower_triangle_mask == 0, -np.inf, inner)

        # the math formula (^-^)
        exp_score = np.exp(inner)
        prob = exp_score / np.sum(exp_score, axis=1, keepdims=True)

        out = prob @ V

        print(prob)
        return out

In [9]:
# Create dummy input (4 words, embedding size 8)
x = np.random.randn(4, 8)
sa = SelfAttention(8, 16)
out = sa.forward(x)

# Let's peek at the probabilities inside (we need to return 'prob' from forward to see this)
# If you print 'prob', you will see:
# [[1.0, 0.0, 0.0, 0.0],  <- Word 1 sees only itself
#  [0.4, 0.6, 0.0, 0.0],  <- Word 2 sees 1 and 2
#  [0.2, 0.2, 0.6, 0.0],  <- Word 3 sees 1, 2, 3
#  [0.1, 0.1, 0.1, 0.7]]  <- Word 4 sees everyone

[[1.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [9.99492990e-01 5.07009873e-04 0.00000000e+00 0.00000000e+00]
 [4.05120304e-07 1.25391616e-10 9.99999595e-01 0.00000000e+00]
 [8.77477236e-01 1.22461193e-01 5.69832900e-05 4.58835133e-06]]


In [2]:
## timme to convert it into pytorch

import torch
import torch.nn as nn 
from torch.nn import functional as F 

In [20]:
class RoPE(nn.Module):
    def __init__(self, dim, theta=10000.0):
        super().__init__()
        self.dim = dim
        self.theta = theta
        # self.freq_cos = 0
        # self.freq_sin = 0
    
    def precompute_freq(self, end):
        freq = 1.0/ (self.theta ** (torch.arange(0,self.dim,2)[: (self.dim//2)].float() / self.dim))
        t = torch.arange(end, device=freq.device)
        freq = torch.outer(t, freq).float()
        freq = torch.cat((freq, freq), dim = -1)
        return freq.cos(), freq.sin()
    
    def rotary_emd(self, x, freq_cos, freq_sin):
        Batch, Time_step, dim_head = x.shape
        half_dim = dim_head // 2
        x1 = x[..., : half_dim]
        x2 = x[..., half_dim: ]

        cos = freq_cos[: Time_step, :half_dim]
        sin = freq_sin[: Time_step, :half_dim]

        output1 = (x1 * cos) - (x2 * sin)
        output2 = (x1 * sin) + (x2 * cos)
        return torch.cat((output1, output2), dim=-1)


In [21]:
class Head(nn.Module):
    def __init__(self, embed_in, head_size):
        super().__init__()

        # For Defining Linear Layers
        self.key = nn.Linear(embed_in, head_size, bias=False)
        self.query = nn.Linear(embed_in, head_size, bias=False)
        self.value = nn.Linear(embed_in, head_size, bias=False)

        self.rope = RoPE(head_size)
        cos, sin = self.rope.precompute_freq(1024)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)
        # mask buffer (100x100) and slice it later to fit the current seq len
        self.register_buffer('tril', torch.tril(torch.ones(1024,1024)))
    
    def forward(self, x):
        Batch_size, Time_step, Channels = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        q = self.rope.rotary_emd(q, self.cos, self.sin)
        k = self.rope.rotary_emd(k, self.cos, self.sin)
        
        # (-2,-1) to flip time_steps and head_size
        inner = q @ k.transpose(-2,-1) * (k.shape[-1]**-0.5)
        #masked trill
        inner = inner.masked_fill(self.tril[:Time_step, :Time_step] == 0, float('-inf'))
        inner = F.softmax(inner, dim=-1)
        out = inner @ v
        return out


In [22]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, embed_in):
        super().__init__()
        self.heads = nn.ModuleList([Head(embed_in, head_size) for _ in range(num_heads)])
        #final projection layer
        self.proj = nn.Linear(num_heads*head_size, embed_in)
    
    def forward(self, x):
        out = [h(x) for h in self.heads]
        out = torch.cat(out, dim=-1)
        out = self.proj(out)
        return out

In [23]:
class RMSNorm(nn.Module):
    def __init__(self, dim, epsilon = 1e-6):
        super().__init__()
        self.epsilon = epsilon
        self.weights = nn.Parameter(torch.ones(dim))
    
    def norm(self, x):
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.epsilon)
        return x
    
    def forward(self, x):
        output = self.norm(x.float()).type_as(x)
        return output*self.weights

In [24]:
class SwiGLU(nn.Module):
    def __init__(self, dim, h_dim, multiple_of = 256):
        super().__init__()
        self.w1 = nn.Linear(dim, h_dim, bias=False)
        self.w2 = nn.Linear(h_dim, dim, bias = False)
        self.w3 = nn.Linear(dim, h_dim, bias=False)
    
    def forward(self, x):
        gate_projection = F.silu(self.w1(x)) #f.silu is torch's swish function
        Value_projection = self.w3(x)
        output_projection = self.w2(gate_projection * Value_projection)
        return output_projection

In [25]:
# class FeedForward(nn.Module):
#     def __init__(self, embed_in):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(embed_in, 4*embed_in),
#             SwiGLU(4*embed_in, 4*4*embed_in),
#             nn.Linear(4*embed_in, embed_in)
#         )
    
#     def forward(self, x):
#         return self.net(x)
class Block(nn.Module):
    def __init__(self, embed_in, num_heads):
        super().__init__()
        head_size = embed_in // num_heads
        h_dim = 4*embed_in
        self.sa = MultiHeadAttention(num_heads, head_size, embed_in)
        # self.ffwd = FeedForward(embed_in)
        self.ffwd = SwiGLU(embed_in, h_dim)
        self.layer_norm1 = RMSNorm(embed_in)
        self.layer_norm2 = RMSNorm(embed_in)
    
    def forward(self, x):
        #the residul connections is "+" sign
        x = x + self.sa(self.layer_norm1(x))
        x = x + self.ffwd(self.layer_norm2(x))
        return x
        

In [26]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, num_emb, num_layers, block_size):
        super().__init__()
        self.token_embed_tab = nn.Embedding(vocab_size, num_emb)
        self.postional_emb_tab = nn.Embedding(block_size, num_emb)
        self.blocks = nn.Sequential(
            *[Block(num_emb, num_heads=4) for _ in range(num_layers)]
        )
        self.layer_norm = nn.LayerNorm(num_emb)
        self.linear_head = nn.Linear(num_emb, vocab_size)
        self.block_size = block_size
    
    def forward(self, idx, targets=None):
        Batch, time_steps = idx.shape

        token_emb = self.token_embed_tab(idx)
        pos_emb = self.postional_emb_tab(torch.arange(time_steps, device=idx.device))
        x = token_emb + pos_emb

        x = self.blocks(x)
        x = self.layer_norm(x)
        logits = self.linear_head(x)
        if targets is None:
            loss = None
        else:
            Batch, time_steps, channel = logits.shape
            logits = logits.view(Batch*time_steps, channel)
            targets = targets.view(Batch*time_steps)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    #this is AI generated:
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # 1. Crop idx to the last 'block_size' tokens
            # (Because our positional embeddings only go up to block_size)
            idx_cond = idx[:, -self.block_size:]
            
            # 2. Get predictions
            logits, loss = self(idx_cond)
            
            # 3. Focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            
            # 4. Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            
            # 5. Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            
            # 6. Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
            
        return idx

In [30]:
class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
    
    def train(self, text, num_merges):
        ids = list(text.encode("utf-8"))

        for i in range(num_merges):
            stats = {}
            for pair in zip(ids, ids[i:]):
                stats[pair] = stats.get(pair, 0) + 1
            
            if not stats:
                break
            best_pair = max(stats, key=stats.get)
            idx = 256 + i
            self.merges[best_pair] = idx
            self.vocab[idx] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            newids = []
            j = 0
            while j < len(ids):
                if j < len(ids) - 1 and ids[j] == best_pair[0] and ids[j+1] == best_pair[1]:
                    newids.append(idx)
                    j += 2
                else:
                    newids.append(ids[j])
                    j += 1
            ids = newids
            print(f"merge {i+1}/{num_merges}: {best_pair} -> {idx}")
    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = {}
            for pair in zip(ids, ids[1:]):
                stats[pair] = stats.get(pair, 0) + 1
            
            can_merge = {p: self.merges[p] for p in stats if p in self.merges}

            if not can_merge:
                break

            pair_to_merge = min(can_merge, key=lambda k : self.merges[k])
            idx = self.merges[pair_to_merge]
            newids = []
            j = 0

            while j < len(ids):
                if j < len(ids) - 1 and ids[j] == pair_to_merge[0] and ids[j+1] == pair_to_merge[1]:
                    newids.append(idx)
                    j += 2
                else:
                    newids.append(ids[j])
                    j += 1
            ids = newids
        return ids
    
    def decode(self, ids):
        tokens = b"".join(self.vocab[idx] for idx in ids)
        return tokens.decode("utf-8" ,errors="replace")

In [32]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import requests # Need this to download the dataset

# --- 0. Hyperparameters ---
batch_size = 32
block_size = 64     # Increased context length slightly
max_iters = 1000
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# BPE Settings
vocab_size = 256 + 5000 # 256 raw bytes + 100 merges

# --- 1. Load Real Data (Tiny Shakespeare) ---
print("Downloading TinyShakespeare...")
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
print(f"Text length: {len(text)} characters")

# --- 2. Train Tokenizer ---
print("Training BPE Tokenizer...")
tokenizer = BPETokenizer()
tokenizer.train(text, num_merges=5000) # Learn 100 new tokens

# Use the tokenizer methods instead of lambdas
encode = tokenizer.encode
decode = tokenizer.decode

# Check if it worked
print(f"Vocab size: {len(tokenizer.vocab)}")
encoded_sample = encode("Hello World")
print(f"Sample encode: 'Hello World' -> {encoded_sample}")
print(f"Sample decode: {decode(encoded_sample)}")

# --- 3. Prepare Data for GPT ---
print("Encoding full dataset...")
# This might take a second because your Python encode loop is not optimized
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Training tokens: {len(train_data)}")

# --- 4. Model & Data Loader ---
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# Ensure your GPTLanguageModel class is defined in this file or imported!
# Passing the NEW vocab size (356)
model = GPTLanguageModel(len(tokenizer.vocab), n_embd, n_layer, block_size)
m = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# --- 5. Training Loop ---
print(f"Training on {device}...")
for iter in range(max_iters):
    if iter % 100 == 0:
        xb, yb = get_batch('train')
        logits, loss = model(xb, yb)
        print(f"Step {iter}: Loss {loss.item():.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# --- 6. Generate Text ---
print("\n--- GENERATED TEXT ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
# Generating 500 tokens (which is more words than 500 chars!)
generated_ids = m.generate(context, max_new_tokens=500)[0].tolist()
print(decode(generated_ids))

Text length: 1115394 characters
Training BPE Tokenizer...
merge 1/5000: (32, 32) -> 256
merge 2/5000: (101, 32) -> 257
merge 3/5000: (32, 111) -> 258
merge 4/5000: (32, 32) -> 259
merge 5/5000: (32, 32) -> 260
merge 6/5000: (32, 32) -> 261
merge 7/5000: (32, 32) -> 262
merge 8/5000: (32, 32) -> 263
merge 9/5000: (32, 32) -> 264
merge 10/5000: (32, 32) -> 265
merge 11/5000: (32, 32) -> 266
merge 12/5000: (32, 32) -> 267
merge 13/5000: (32, 32) -> 268
merge 14/5000: (32, 32) -> 269
merge 15/5000: (32, 32) -> 270
merge 16/5000: (32, 32) -> 271
merge 17/5000: (32, 32) -> 272
merge 18/5000: (32, 32) -> 273
merge 19/5000: (32, 32) -> 274
merge 20/5000: (32, 32) -> 275
merge 21/5000: (32, 32) -> 276
merge 22/5000: (32, 32) -> 277
merge 23/5000: (32, 32) -> 278
merge 24/5000: (32, 32) -> 279
merge 25/5000: (32, 32) -> 280
merge 26/5000: (32, 32) -> 281
merge 27/5000: (32, 32) -> 282
merge 28/5000: (32, 32) -> 283
merge 29/5000: (32, 32) -> 284
merge 30/5000: (32, 32) -> 285
merge 31/5000: (32,

In [28]:
#training code is generated by AI as i got a little smooth brained.

import torch
import torch.nn as nn
from torch.nn import functional as F

# --- Hyperparameters ---
batch_size = 32
block_size = 8 # Context length
max_iters = 1000
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32
n_head = 4
n_layer = 3
dropout = 0.0

# --- 1. Load Data ---
text = """
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.
"""
# Create Vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Train/Test Split
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

# --- 2. Data Loader ---
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# --- 3. Initialize Model ---
model = GPTLanguageModel(vocab_size, n_embd, n_layer, block_size)
m = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# --- 4. Training Loop ---
print(f"Training on {device}...")
for iter in range(max_iters):

    # Every once in a while evaluate the loss on train and val sets
    if iter % 100 == 0:
        xb, yb = get_batch('train')
        logits, loss = model(xb, yb)
        print(f"Step {iter}: Loss {loss.item():.4f}")

    # Sample a batch of data
    xb, yb = get_batch('train')

    # Evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# --- 5. Generate Text ---
print("\n--- GENERATED TEXT ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=1024)[0].tolist()))

Training on cpu...
Step 0: Loss 3.7645
Step 100: Loss 1.5822
Step 200: Loss 0.5811
Step 300: Loss 0.3608
Step 400: Loss 0.3012
Step 500: Loss 0.2921
Step 600: Loss 0.3185
Step 700: Loss 0.3113
Step 800: Loss 0.3062
Step 900: Loss 0.3132

--- GENERATED TEXT ---

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved. resolved. resolved rather to die than to famish?


All:
Speak, speak.

First Citizen:
First, you know Caius ue Marcius is cius is cius is cius is cius is cius is cius is cius is cius is cius is cius is cistizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved. resolved rather to die than to famish?

All:
Spemmer, hear me speak.

All:
Speak, speak.

First Citizen:
First, you know Caius Marcius is cius is cius is cisolved. resolved rather to die than to famish?

All:
Speak, speak.

All:
fore we proceed any further, hear me speak.

All:
Speak, speak.

All:
Speak, speak.

First Citizen:
B